# 02.6 — `nn.Module` vs `layerGraph`

MATLAB assembles networks as a **static graph object** you build up front. PyTorch defines them as **classes** — declare the parts in `__init__`, wire them together with ordinary Python in `forward`. Everything else about working with models (parameters, checkpoints, train/eval modes, composition) follows from that one design choice.

This notebook covers:

* The `nn.Module` contract: `__init__` + `forward`, and why `model(x)` works.
* Automatic parameter registration — and the plain-list trap that silently breaks it.
* `nn.Sequential`, `nn.ModuleList`, and composing modules from modules.
* `state_dict` — checkpoints, and how this repo's `optimal_state.pt` works.
* `train()` vs `eval()` modes.

**Prerequisites:** [01.4 classes and OOP](../01_python_for_matlab_users/01.4_classes_and_oop.ipynb), [02.5 autograd](02.5_autograd_basics.ipynb).

## Section 1 — What MATLAB does

The Deep Learning Toolbox builds networks declaratively:

```matlab
layers = [
    sequenceInputLayer(58)
    gruLayer(100, 'OutputMode', 'sequence')
    fullyConnectedLayer(4)
    softmaxLayer
];
lgraph = layerGraph(layers);
lgraph = connectLayers(lgraph, 'gru', 'fc');   % explicit wiring for branches
net = dlnetwork(lgraph);
analyzeNetwork(net)                              % inspect the static graph
```

Layers are named strings; branches and skips are wired with `connectLayers`; the result is a fixed structure. The MATLAB pipeline's `PARAMETERS_cgg_constructNetworkArchitecture` builds exactly such graphs from its 47 architecture options.

PyTorch has no separate graph object — **the code in `forward` IS the architecture**, re-traced each call ([02.5 §2.1](02.5_autograd_basics.ipynb)). Branching, skipping, looping: just Python.

## Section 2 — The Python concepts you need

### 2.1 — The `nn.Module` contract

In [ ]:
import torch
from torch import nn

class TinyGRUClassifier(nn.Module):
    """GRU encoder + linear head — a 10-line sketch of this project's Milestone B model."""

    def __init__(self, in_features: int, hidden: int, num_classes: int):
        super().__init__()                                    # mandatory (01.4)
        self.gru = nn.GRU(in_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x):                                      # x: (B, T, features)
        out, _ = self.gru(x)          # out: (B, T, hidden)
        return self.head(out[:, -1])  # classify from the last timestep

model = TinyGRUClassifier(in_features=8, hidden=16, num_classes=3)
logits = model(torch.randn(4, 10, 8))     # ← calls forward() via __call__
print(logits.shape)

Two contract details:

* **You never call `forward` directly** — `model(x)` goes through `nn.Module.__call__`, which runs hooks and bookkeeping around your `forward`. (This is the `__call__` dunder from [01.4 §2.4](../01_python_for_matlab_users/01.4_classes_and_oop.ipynb) earning its keep.)
* **Assignment registers**: the moment `self.gru = nn.GRU(...)` executes, the GRU becomes a *child module* — its parameters appear in `model.parameters()`, move with `model.to(device)`, and save with the checkpoint. No `connectLayers`, no names-as-strings.

In [ ]:
# The registration tree, for free:
for name, p in model.named_parameters():
    print(f"{name:22} {tuple(p.shape)}")
print("total:", sum(p.numel() for p in model.parameters()))

### 2.2 — The plain-list trap (and `nn.ModuleList`)

Registration happens through attribute assignment of `nn.Module` values — and a **plain Python list of modules is not an `nn.Module` value**. The parameters inside it silently vanish from `parameters()`:

In [ ]:
class Broken(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = [nn.Linear(4, 2) for _ in range(3)]      # plain list — NOT registered!

class Fixed(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList(nn.Linear(4, 2) for _ in range(3))

print("Broken params:", sum(p.numel() for p in Broken().parameters()))   # 0 !!
print("Fixed params: ", sum(p.numel() for p in Fixed().parameters()))

**Silent** is the operative word: `Broken` runs forward fine, the loss computes, `backward()` works — but the optimizer never sees those parameters, so the heads never train. No exception anywhere.

This matters directly here: the production `MultiHeadClassifier` (one output head per target dimension — the 4 quaddle feature dims) stores its heads in exactly this `nn.ModuleList`. `nn.ModuleDict` is the same idea keyed by name; `nn.Sequential` is the special case of "modules applied in order" and covers most simple chains:

In [ ]:
# nn.Sequential — the layerGraph analog for straight-line stacks
mlp = nn.Sequential(
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 3),
)
print(mlp(torch.randn(2, 8)).shape)

### 2.3 — Modules compose: the tree

Because registration recurses, modules nest into arbitrarily deep trees — and the whole tree behaves as one model. This is the architecture pattern of the entire production codebase: build small modules, compose them. Watch the *real* variational composite assemble from a config, exactly as the training CLI does it:

In [ ]:
from neural_data_decoding.models.composite import build_variational_composite

cfg = {
    "in_features": 8, "samples_per_window": 1, "num_areas": 1,
    "hidden_sizes": [16, 4],            # encoder 16 → latent 4
    "num_classes_per_dim": [3],
    "classifier_hidden_size": [8],
    "transform": "GRU",
    "dropout": 0.0, "want_normalization": False, "activation": "",
    "loss_type_decoder": "MSE", "classifier_dropout": 0.5,
    "confidence_type": [], "stitching_and_fusion_layer": "",
}
composite = build_variational_composite(cfg)

print(type(composite).__name__, "—", sum(p.numel() for p in composite.parameters()), "parameters")
print("\ntop-level children:")
for name, child in composite.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"  {name:14} {type(child).__name__:24} {n:6d} params")

Eight children — NaN handling, shape adapters, encoder, bottleneck, sampling, decoder, classifier — each itself a module tree. The composite's `forward` wires them with plain Python (you read it in [02.2 §3](02.2_array_axis_conventions.ipynb)); MATLAB expresses the same topology as ~40 `connectLayers` calls in `cgg_constructNetworkArchitecture`.

### 2.4 — `state_dict`: checkpoints

A module's `state_dict()` is an ordered dict of every parameter (and buffer) by pathname — the portable snapshot of a trained model:

In [ ]:
sd = model.state_dict()
print(list(sd.keys()))

# Save / load round-trip — this is all a checkpoint is:
torch.save(sd, "/tmp/demo_ckpt.pt")
model2 = TinyGRUClassifier(in_features=8, hidden=16, num_classes=3)
model2.load_state_dict(torch.load("/tmp/demo_ckpt.pt", weights_only=True))
x = torch.randn(2, 10, 8)
print("identical outputs:", torch.equal(model(x), model2(x)))

Notes worth pinning:

* `load_state_dict` requires the **same architecture** — keys and shapes must match. (The keys are the attribute pathnames; rename `self.gru` to `self.encoder` and old checkpoints stop loading.)
* `weights_only=True` is the safe-loading flag for untrusted files.
* This repo's `optimal_state.pt` / `current_state.pt` outputs are exactly this, plus optimizer state — and the Stage 1 → Stage 2 handoff (`copy_autoencoder_weights`) is state-dict surgery between two composites.

### 2.5 — `train()` vs `eval()` modes

Some layers behave differently during training vs inference — dropout drops, batch-norm updates running statistics. One flag on the tree controls all of them:

In [ ]:
dropout_model = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5))
x = torch.ones(1, 4)

dropout_model.train()
print("train mode:", dropout_model(x))    # ~half the values zeroed, rest scaled ×2

dropout_model.eval()
print("eval mode: ", dropout_model(x))    # deterministic, dropout off

Forgetting `model.eval()` before validation is the classic silent bug: metrics get computed through active dropout and look noisily worse than the model really is. The production training loop flips modes around every validation pass — and this project's VAE **sampling layer** keys off the same flag (stochastic latents in train, deterministic `z = mu` at eval; the deep dive is planned as notebook 06.13).

## Section 3 — The `neural_data_decoding` implementation

The `MultiHeadClassifier` — `nn.ModuleList` from §2.2, in production:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/classifier.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "ModuleList" in line)
for k in range(max(0, i - 10), min(i + 6, len(src))):
    marker = " ►" if "ModuleList" in src[k] else "  "
    print(f"{k + 1:4}{marker}| {src[k]}")

Things to spot:

* One `nn.Linear` head per output dimension, collected in an `nn.ModuleList` — so `parameters()` sees all of them and the optimizer trains all of them.
* The heads are built with a comprehension ([01.2 §2.6](../01_python_for_matlab_users/01.2_control_flow.ipynb)) — module construction is just Python.
* `forward` returns a **list** of per-dimension logits; the multi-head cross-entropy loss consumes it downstream.

## Section 4 — Hands-on exercises

### Exercise 4.1 — find the bug

The model below trains, loss decreases… but only partly. What never learns, and why?

In [ ]:
class Suspicious(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Linear(8, 16)
        self.extra_layers = [nn.Linear(16, 16), nn.Linear(16, 16)]
        self.head = nn.Linear(16, 2)

    def forward(self, x):
        h = torch.relu(self.backbone(x))
        for layer in self.extra_layers:
            h = torch.relu(layer(h))
        return self.head(h)

print("registered params:", sum(p.numel() for p in Suspicious().parameters()))
print("actual params    :", sum(p.numel() for m in [Suspicious().backbone, *Suspicious().extra_layers, Suspicious().head] for p in m.parameters()))

`extra_layers` is a plain list — its two Linears (2 × 272 params) exist and run in `forward`, but they're invisible to `parameters()`, so the optimizer never updates them. Fix: `nn.ModuleList([...])`.

### Exercise 4.2 — count the composite

Using the `composite` built in §2.3: which of its eight children holds the most parameters? Answer with one loop (no peeking at the §2.3 output).

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
biggest = max(composite.named_children(), key=lambda nc: sum(p.numel() for p in nc[1].parameters()))
print(biggest[0], "-", sum(p.numel() for p in biggest[1].parameters()), "params")

## Section 5 — Common errors

### Optimizer runs, some layers never improve

The plain-list trap (§2.2 / Exercise 4.1). Audit with: does `sum(p.numel() for p in model.parameters())` match what you'd expect from the architecture?

### `AttributeError: cannot assign module before Module.__init__() call`

Missing `super().__init__()` — PyTorch's guard, covered in [01.4 §2.3](../01_python_for_matlab_users/01.4_classes_and_oop.ipynb).

### `RuntimeError: Error(s) in loading state_dict ... Missing key(s) / Unexpected key(s)`

Architecture drift between save and load — renamed attributes, changed layer counts, or a `DataParallel`-wrapped save (`module.`-prefixed keys). Compare `list(sd.keys())` on both sides; the mismatch names itself.

### Validation metrics noisy / worse than training

`model.eval()` was never called — dropout is still active during evaluation. (And remember its partner: wrap the loop in `torch.no_grad()`, [02.5 §2.4](02.5_autograd_basics.ipynb).)

### Defining layers inside `forward`

`self.fc = nn.Linear(...)` inside `forward` creates FRESH weights every call — nothing accumulates training. Layers are declared once in `__init__`; `forward` only *uses* them.

### `NotImplementedError` when calling the model

Your `forward` is misspelled (`foward`, `Forward`) — `nn.Module`'s default `forward` raises this.

## Section 6 — Further reading

- [nn.Module docs](https://pytorch.org/docs/stable/generated/torch.nn.Module.html) — the full API.
- [Building models tutorial](https://pytorch.org/tutorials/beginner/introyt/modelsyt_tutorial.html) — official walkthrough.
- [Saving & loading models](https://pytorch.org/tutorials/beginner/saving_loading_models.html) — state_dict best practices.
- [`src/neural_data_decoding/models/composite.py`](../../src/neural_data_decoding/models/composite.py) — the production module-of-modules.

Next up: **[02.7 optimizers and learning rates](02.7_optimizers_and_learning_rates.ipynb)**.